#Fusion Transcript Detection in Rare Disease Tutorial

For the workshop on fusion transcript detection we will analyze RNA fusion data from a young male rare disease patient suffering from outgrowths of the bones (a condition known as multiple exostoses).  DNA-based testing (trio sequencing, clinical aCGH and MLPA of the EXT1 and EXT2 genes, which are mutated in most cases of multiple exostoses) revealed no genetic findings of interest.  RNA-Seq has been performed on patient whole-blood and fusions have been called using a fusion calling algorithm.  The tutorial focuses on prioritizing results of potential biological relevance to the patient's condition.



---



![alt text](https://onlinelibrary.wiley.com/cms/asset/a2372cb2-0add-4b48-a1f5-886360ca61e8/mgg3560-fig-0002-m.jpg)




---



#Download required files and install necessary packages
We will start by downloading some required files and installing a few dependencies.  

The files include fusion candidates for analysis as well as various reference files and basic databases we will use later.  

The first three packages provide us with some useful general functions that make for cleaner code, while the final three install the PCAN (Phenotype Consensus ANalysis) library - a key tool for prioritizing results of phenotypic relevance while enabling us to minimize use of arbitrary filters that can remove important results.

Now - either click within the box below and hit CTRL+ENTER **or** click on the play icon in the top left of the box.  Then watch as the code executes and the outputs for this code block are generated beneath.

You will do this for every code block in the tutorial in consecutive order (except for one, which is identified later).

Give it a go, and remember to wait for the swirling around the icon to cease before running any other blocks of code.

In [ ]:
# Download necessary files
#system('wget --recursive --no-parent --reject="index.html*" -nH --cut-dirs=3 path removed')

# Install R packages
install.packages("tidyr")
install.packages("dplyr")
install.packages("stringr")
install.packages("BiocManager")
BiocManager::install("PCAN")
BiocManager::install("org.Hs.eg.db")

print("Complete - proceeed to next code block")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'?repositories' for details

replacement repositories:
    CRAN: https://cran.rstudio.com


Bioconductor version 3.13 (BiocManager 1.30.16), R 4.1.0 (2021-05-18)

Installing package(s) 'BiocVersion', 'PCAN'

also installing the dependencies ‘formatR’, ‘lambda.r’, ‘futile.options’, ‘futile.logger’, ‘snow’, ‘BiocParallel’


Old packages: 'broom', 'colorspace', 'curl', 'dplyr', 'gert', 'ggplot2',
  'mime', 'openssl', 'rmarkdown', 'testthat', 'xfun'

'getOption("repos")' replaces Bioconductor standard repositories, see
'?repositories' for details

replacement repositories:
    CRAN: https://cran

[1] "Complete - proceeed to next code block"


# Getting started

With the required dependencies successfully installed, we can get down to work.

Our first step is to load some candidate fusion events into memory.  These have previously been generated by running a fusion transcript detection algorithm on patient RNA-Seq data.  Many solutions exist e.g. STAR-Fusion, Tophat Fusion, FusionCatcher and more.  While we have had good success with Tophat Fusion, others may have their own preferences.  Thus, for the purposes of this tutorial we use a generic input format that can readily be generated from the outputs of any fusion detection algorithm with minimal processing.

#Input data

Fusion detection algorithms filter candidate fusions to reduce the number of results they output.  While the filters they employ aim to remove false positive results they often remove many real results due to limitations of the data sets and methods used to train the filters.  

We advocate use of the UNfiltered fusions from any algorithm, followed by a more interactive review and prioritization process.   First, let's load the filtered and unfiltered results together though...

In [ ]:
# Read unfiltered fusions into table
fus_unfilt_in<-read.table("generic_exostoses_fusions_unfiltered.tsv", header=TRUE, stringsAsFactors=FALSE)
# Read filtered fusions into table
fus_filt_in<-read.table("generic_exostoses_fusions_Filtered.tsv", header=TRUE, stringsAsFactors=FALSE)
# Preview unfiltered fusion table 
head(fus_unfilt_in)

,FusionName,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint
,<chr>,<int>,<int>,<chr>,<chr>
1,IGKV4-1--AC096579.7,955,11544,chr2:89185670:+,chr2:89161433:-
2,IGKV4-1--IGKC,328,11529,chr2:89185670:+,chr2:89161433:-
3,IGKV4-1--AC096579.7,293,11544,chr2:89185664:+,chr2:89161439:-
4,IGLV2-23--IGLL5,86,12193,chr22:23040904:+,chr22:23235963:+
5,IGLV2-23--IGLC1,69,12186,chr22:23040904:+,chr22:23235963:+
6,IGLV2-23--IGLL5,9,12193,chr22:23040892:+,chr22:23237555:+


First we load the files and then have a look at our generic input format.  You can see it is very minimalistic, consisting only of fusion identifers (GeneA--GeneB format), the number of reads supporting each fusion candidate, and the chromosomal positions corresponding to the points of fusion.

(As an aside - note how many candidates involve Immunoglobulin (IG*) genes.  We will come back to this later.)


#Filtered vs unfiltered input numbers

Next lets have a look at how many results each file contains...

In [ ]:
# Output number of rows in unfiltered and filtered fusion tables respectively
nrow(fus_unfilt_in)
nrow(fus_filt_in)

[1] 31036

[1] 257

As you can see, there is a pretty big difference!  

It's easy to see why fusion detection algorithms usually filter their outputs heavily.  Tens of thousands of results can be extremely overwhelming to consider.  

However, the filters used by fusion calling algortihms to reduce these numbers are often trained using well known cancer fusions.  Would we expect that the evidence supporting a highly expressed oncogenic fusion would appear similar to that supporting a loss-of-function event in a rare disease?  Not necessarily.  Their nature and mechanisms are likely very different, and this could well mean differences in their patterns of read support and detectability.  Hence we choose to omit these filters and instead opt to deal with the initially overwhelming output of over 30k results. 

There are intuitive and biologically mindful ways to navigate the data however, so don't despair.  Let's keep going.


#Some new columns + read support in filtered and unfiltered inputs

In [ ]:
# Load tidyr library to utilize 'separate' function
library(tidyr)
# Split fusion name into gene names
fus_unfilt_in<-separate(fus_unfilt_in, FusionName, c("GeneA", "GeneB"), sep="--", remove=FALSE)
# Create a total count of supporting reads per fusion from the 'junction' and 'spanning' reads
fus_unfilt_in$TotalCount=fus_unfilt_in$JunctionReadCount+fus_unfilt_in$SpanningFragCount
# Preview the new table
head(fus_unfilt_in)

,FusionName,GeneA,GeneB,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>
1,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,955,11544,chr2:89185670:+,chr2:89161433:-,12499
2,IGKV4-1--IGKC,IGKV4-1,IGKC,328,11529,chr2:89185670:+,chr2:89161433:-,11857
3,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,293,11544,chr2:89185664:+,chr2:89161439:-,11837
4,IGLV2-23--IGLL5,IGLV2-23,IGLL5,86,12193,chr22:23040904:+,chr22:23235963:+,12279
5,IGLV2-23--IGLC1,IGLV2-23,IGLC1,69,12186,chr22:23040904:+,chr22:23235963:+,12255
6,IGLV2-23--IGLL5,IGLV2-23,IGLL5,9,12193,chr22:23040892:+,chr22:23237555:+,12202


All we have done is to create a total supporting read count (TotalCount) from the two separate read support values (JunctionReadCount & SpanningFragCount), and to separate the fusion identifier (FusionName) into individual gene IDs (GeneA & GeneB) that can be used as input to PCAN a little later.

Before we forget about the filtered fusion file for good, let's generate a total read count for it too, and compare to the unfiltered file...

In [ ]:
# Summarize the distribution of the total read counts for each unfiltered fusions
summary(fus_unfilt_in$TotalCount)
# Create the totalcount column and summarize the distribution of the total read counts for each filtered fusions
fus_filt_in$TotalCount=fus_filt_in$JunctionReadCount+fus_filt_in$SpanningFragCount
summary(fus_filt_in$TotalCount)

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
    1.00     1.00     2.00    21.94     5.00 12499.00 

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
    8.00    10.00    15.00    82.48    30.00 11551.00 

You can see that the total supporting read count for the filtered fusions tends to be higher.  Obviously there are a lot of fusions in the unfiltered file with only one or two supporting reads.  It would be understandable to think that fusions with such low read support could be safely discarded.  

What if formation of a fusion transcript led to the introduction of a stop codon and transcript degradation through nonsense-mediated decay though?  What if the fusion originated in a different tissue than the one we are testing?  How many supporting reads would be enough?  It is very hard to say with certainty.  Thus, while it is tempting to apply a minimum read support filter, let's resist the urge and look for another route.

---

##CHECKPOINT 1

---



# Highly expressed genes of limited biological interest
First, recall how many fusions appeared to involve immunoglobulin genes.  Fusion candidates often disproportionately involve genes of limited phenotypic interest that are the most highly expressed genes in the tissue being tested.  In our case, whole blood was sequenced and experience has taught us that many candidates arise from hemoglobin and immunoglobulin genes.  Since our patient does not have a blood-based or immune disorder, we will deprioritize such events.  Nonetheless, we won't remove such candidates entirely, just in case one should turn out to involve a second gene that IS of interest.

We will create two new columns that tells us if a fusion involves one or two hemoglobin/immunoglobulin genes and count those candidates.

In [ ]:
# Load the stringr and dplyr libraries to access the 'str_detect' and 'distinct' functions
library(stringr)
library(dplyr)

# Count number of unfiltered fusions
nrow(fus_unfilt_in)

# Create table of fusions where BOTH genes in a fusion are hemoglobin or immunoglobulin genes
HB_IG_both<-bind_rows(filter(fus_unfilt_in,str_detect(GeneA, "^HB")), filter(fus_unfilt_in,str_detect(GeneB, "^HB")), 
                filter(fus_unfilt_in,str_detect(GeneA, "^IG")), filter(fus_unfilt_in,str_detect(GeneB, "^IG")))

# Remove duplicate rows from 'both' table
HB_IG_both<-distinct(HB_IG_both)

# Create table of fusions where EITHER gene in a fusion is a hemoglobin or immunoglobulin gene
HB_IG_either<-bind_rows(filter(fus_unfilt_in, (str_detect(GeneA, "^HB") | str_detect(GeneA, "^IG")) 
                 & (str_detect(GeneB, "^HB") | str_detect(GeneB, "^IG"))))

# Remove duplicate rows from 'either' table
HB_IG_either<-distinct(HB_IG_either)

# Count number of rows in each table
nrow(HB_IG_both)
nrow(HB_IG_either)

# Annotate original unfiltered fusion table with this new information
fus_unfilt_in$HB_IG_both<-fus_unfilt_in$FusionName %in% HB_IG_both$FusionName
fus_unfilt_in$HB_IG_either<-fus_unfilt_in$FusionName %in% HB_IG_either$FusionName

# Preview the new unfiltered fusion table
head(fus_unfilt_in)
   




Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] 31036

[1] 7491

[1] 1156

,FusionName,GeneA,GeneB,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount,HB_IG_both,HB_IG_either
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<lgl>,<lgl>
1,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,955,11544,chr2:89185670:+,chr2:89161433:-,12499,TRUE,FALSE
2,IGKV4-1--IGKC,IGKV4-1,IGKC,328,11529,chr2:89185670:+,chr2:89161433:-,11857,TRUE,TRUE
3,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,293,11544,chr2:89185664:+,chr2:89161439:-,11837,TRUE,FALSE
4,IGLV2-23--IGLL5,IGLV2-23,IGLL5,86,12193,chr22:23040904:+,chr22:23235963:+,12279,TRUE,TRUE
5,IGLV2-23--IGLC1,IGLV2-23,IGLC1,69,12186,chr22:23040904:+,chr22:23235963:+,12255,TRUE,TRUE
6,IGLV2-23--IGLL5,IGLV2-23,IGLL5,9,12193,chr22:23040892:+,chr22:23237555:+,12202,TRUE,TRUE


You can see that around 7.5k candidates contain either a hemoblobin or immunoglobulin gene, while 1100 contain two such genes.  We could perhaps justify removing some or all of these from further consideration but annotating candidates appropriately instead of filtering them empowers us to investigate later if needed.  

Note, the implementation of this step will vary based on the tissue tested and the disorder being considered.  E.g. If testing cardiac muscle for a cardiac disorder, you may need to prioritize the most highly expressed cardiac genes to look for fusions.  

# Internal control database
Now, let's load a database of candidate fusions that have been detected in other unrelated inherited disease cases at our own institution.  This could be useful for identifying common artifacts, polymorphic events or false positives.  Since all of our patients have distinct rare diseases, we can assume it would be unlikely to see the same pathogenic fusion in multiple patients.  

In [ ]:
# Read the internal fusion database into a table from text file
internaldb<-read.table("InternalFusionDB.tsv", header=TRUE, stringsAsFactors=FALSE, comment.char="")
# Count the number of candiate fusions in the table
nrow(internaldb)
# Preview the internal database table
head(internaldb)

[1] 1477032

,Fusion,Occurrences,AvReads
,<chr>,<int>,<dbl>
1,VCAN--ATG10,1,2
2,VCAN#AS1--ATG10,1,2
3,NMNAT3--MIA3,1,1
4,ANKRD28--CAMK2D,1,1
5,LRRFIP1--PDE6C,1,2
6,HLA#E--ZBTB7C,1,1


You can see that this database of only approximately 100 patients contains a lot of (almost 1.5 million!) supposed 'candidates', many of which will not be disease relevant, or even real.  Thus if our candidate fusions have been seen in other patients, we can likely remove them from consideration when reviewing the current patient's data.

We will use this to help explore the data soon.  First, let's load another database...

In [ ]:
# Read the GTEx normal tissue database fusion candidates into a table from text file
gtex<-read.table("gtex_only_fusion_lib.Mar2019.dat", header=FALSE, stringsAsFactors=FALSE, comment.char="", sep="\t", quote="")
# Add column names
colnames(gtex)<-c("FusionName", "DB", "Details")
# Preview the GTEx normal tissue fusions table
head(gtex)

,FusionName,DB,Details
,<chr>,<chr>,<chr>
1,ABHD14B--SORBS2,"[""GTEx_recurrent_StarF2019""]","{""GTEx_recurrent_StarF2019"":""GTEx-Thyroid:0.899;n=4|GTEx-Blood:0.800;n=4|GTEx-Colon:0.600;n=3|GTEx-Breast:1.034;n=3|GTEx-Adipose Tissue:0.600;n=3|GTEx-Skin:0.400;n=2|GTEx-Lung:0.468;n=2|GTEx-Liver:1.143;n=2|GTEx-Spleen:0.617;n=1|GTEx-Prostate:0.658;n=1|GTEx-Pituitary:0.546;n=1|GTEx-Ovary:0.752;n=1|GTEx-Esophagus:0.200;n=1|GTEx-Brain:0.200;n=1|GTEx-Blood Vessel:0.200;n=1|GTEx-Adrenal Gland:0.526;n=1"",""ATTS"":[""GTEx_recurrent_StarF2019""]}"
2,AC000078.1--RPL8,"[""GTEx_recurrent_StarF2019""]","{""GTEx_recurrent_StarF2019"":""GTEx-Nerve:0.242;n=1|GTEx-Lung:0.234;n=1|GTEx-Breast:0.345;n=1|GTEx-Blood:0.200;n=1"",""ATTS"":[""GTEx_recurrent_StarF2019""]}"
3,AC000124.1--ARF5,"[""GTEx_recurrent_StarF2019""]","{""GTEx_recurrent_StarF2019"":""GTEx-Testis:11.197;n=29"",""ATTS"":[""GTEx_recurrent_StarF2019""]}"
4,AC000124.1--FSCN3,"[""GTEx_recurrent_StarF2019""]","{""ATTS"":[""GTEx_recurrent_StarF2019""],""GTEx_recurrent_StarF2019"":""GTEx-Testis:1.544;n=4""}"
5,AC002398.2--TPM2,"[""GTEx_recurrent_StarF2019""]","{""ATTS"":[""GTEx_recurrent_StarF2019""],""GTEx_recurrent_StarF2019"":""GTEx-Muscle:0.400;n=2""}"
6,AC004771.1--ALDOA,"[""GTEx_recurrent_StarF2019""]","{""GTEx_recurrent_StarF2019"":""GTEx-Muscle:4.400;n=22"",""ATTS"":[""GTEx_recurrent_StarF2019""]}"


This database is a subset of the CTAT Human Fusion Library data (part of Trinity Cancer Transcriptome Analysis Toolkit Fusion-finding modules).  The location of the full database is available in the "Notes" file provided earlier.  We have subsetted the full data to contain only fusion canddiates of high confidence that were identified in normal tissues during an analysis of hundreds of recently deceased individuals as part of the GTEx project.  

While there is some nuance to consider here (see our publication https://doi.org/10.3389/fgene.2020.00173), this is a highly filtered dataset and we will assume that pathogenic fusions causing very rare disease like multiple exostoses will not be present with significant evidence in a normal tissue database.  This assumption will be useful later...  

First let's add presence in our internal and GTEx databases as an annotation in our unfiltered fusions table...

In [ ]:
# Annotate the unfiltered fusions that occur in the GTEx normal tissue database
fus_unfilt_in$InGTEx<-fus_unfilt_in$FusionName %in% gtex$FusionName
# Annotate the unfiltered fusions that occur in the internal database
fus_unfilt_in$InInternal<-fus_unfilt_in$FusionName %in% internaldb$Fusion
# Create a column to annotate if a fusion occurs in either of the two databases
fus_unfilt_in$InEither=fus_unfilt_in$InGTEx==TRUE | fus_unfilt_in$InInternal==TRUE
# Preview the table
head(fus_unfilt_in)

,FusionName,GeneA,GeneB,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount,HB_IG_both,HB_IG_either,InGTEx,InInternal,InEither
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
1,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,955,11544,chr2:89185670:+,chr2:89161433:-,12499,TRUE,FALSE,FALSE,FALSE,FALSE
2,IGKV4-1--IGKC,IGKV4-1,IGKC,328,11529,chr2:89185670:+,chr2:89161433:-,11857,TRUE,TRUE,TRUE,FALSE,TRUE
3,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,293,11544,chr2:89185664:+,chr2:89161439:-,11837,TRUE,FALSE,FALSE,FALSE,FALSE
4,IGLV2-23--IGLL5,IGLV2-23,IGLL5,86,12193,chr22:23040904:+,chr22:23235963:+,12279,TRUE,TRUE,FALSE,FALSE,FALSE
5,IGLV2-23--IGLC1,IGLV2-23,IGLC1,69,12186,chr22:23040904:+,chr22:23235963:+,12255,TRUE,TRUE,FALSE,FALSE,FALSE
6,IGLV2-23--IGLL5,IGLV2-23,IGLL5,9,12193,chr22:23040892:+,chr22:23237555:+,12202,TRUE,TRUE,FALSE,FALSE,FALSE


You can see we now have three new columns providing TRUE/FALSE values for if a fusion is seen in the internal sample database, the GTEx fusion database, or in either of the two.

If we planned to filter fusions that appeared in either database, how many would be removed?

In [ ]:
# Count the total number of unfiltered fusions
nrow(fus_unfilt_in)
# Count how many fusions appear in neither the internal nor GTEx database
sum(fus_unfilt_in$InEither=="FALSE")

[1] 31036

[1] 25208

Approximately 6k candidate fusions could be removed in this manner if we wished.  Let's consider another annotation that can be useful in helping us to prioritize real biological phenomena over artifacts.

#Fusions corresponding to known exon boundaries
Often, a mature fusion transcript will fuse at known exon boundaries.  This is because fusions often occur due to breakage/joining within the long intronic regions of the genome.  Splicing mechanisms subsequently lead to removal of the intronic sequence and so the mature mRNA fusion transcript appears to have fused perfectly at exon boundaries.  This is not always the case, but it is frequently so, and when we see precise exon-exon joining, it enables increased confidence in the true biological nature of the fusion transcript candidate.  

Using a file of known exon boundaries downloaded from the UCSC table browser, we can quickly annotate fusions corresponding to either one or two known exon boundaries.

In [ ]:
# Load into a table from texfile a list of exon boundaries downloaded from the UCSC table browser
exon_boundaries<-read.table("formatted_exons_ucsc_hg19_5_2020.bed", header=FALSE, stringsAsFactors=FALSE, col.names = c("LeftBoundary", "RightBoundary"))
# Preview the exon boundary table
head(exon_boundaries)

,LeftBoundary,RightBoundary
,<chr>,<chr>
1,chr1:11874:+,chr1:12227:+
2,chr1:12613:+,chr1:12721:+
3,chr1:13221:+,chr1:14409:+
4,chr1:11874:+,chr1:12227:+
5,chr1:12646:+,chr1:12697:+
6,chr1:13221:+,chr1:14409:+


With the exon boundaries in the same format as those in our unfiltered fusion files, we can quickly add this new annotation and determine how many fusion candidates correspond to known exon boundaries.

In [ ]:
# Annotate whether the fusion boundaries coincide with known exon boundaries
fus_unfilt_in$LeftKnown<-fus_unfilt_in$LeftBreakpoint %in% exon_boundaries$LeftBoundary
fus_unfilt_in$RightKnown<-fus_unfilt_in$RightBreakpoint %in% exon_boundaries$RightBoundary
# Preview the newly annotated table
head(fus_unfilt_in)
# Count the number of fusions for which both fusion boundaries coincide with known exon boundaries
nrow(fus_unfilt_in[which(fus_unfilt_in$RightKnown=="TRUE" & fus_unfilt_in$LeftKnown=="TRUE"),])

,FusionName,GeneA,GeneB,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount,HB_IG_both,HB_IG_either,InGTEx,InInternal,InEither,LeftKnown,RightKnown
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
1,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,955,11544,chr2:89185670:+,chr2:89161433:-,12499,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
2,IGKV4-1--IGKC,IGKV4-1,IGKC,328,11529,chr2:89185670:+,chr2:89161433:-,11857,TRUE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE
3,IGKV4-1--AC096579.7,IGKV4-1,AC096579.7,293,11544,chr2:89185664:+,chr2:89161439:-,11837,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE
4,IGLV2-23--IGLL5,IGLV2-23,IGLL5,86,12193,chr22:23040904:+,chr22:23235963:+,12279,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE
5,IGLV2-23--IGLC1,IGLV2-23,IGLC1,69,12186,chr22:23040904:+,chr22:23235963:+,12255,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE
6,IGLV2-23--IGLL5,IGLV2-23,IGLL5,9,12193,chr22:23040892:+,chr22:23237555:+,12202,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE


[1] 238

Only 238 fusion candidates occur at two known exon boundaries.  This information could be useful later, but first let's add perhaps our most useful and novel annotation.

---
###CHECKPOINT 2

---

#PCAN (Phenotype Consensus ANalysis)

PCAN takes a list of standardized phenotype terms relevant to a patient's condition in the format of the Human Phenotype Ontology (https://hpo.jax.org/app/).  It also takes as input an HGNC gene ID and identifies phenotype terms known to be associated with the gene utilizing information from the ClinVar and OMIM databases.  PCAN calculates a semantic similarity score that provides a measure of how similar the patient phenotype terms are to the phenotype terms associated with the gene.  It does the same for all genes known to be associated with disease in the ClinVar database, and than ranks the gene being tested against all of these other genes, in terms of semantic similarity between phenotypes.  

**In simpler terms** - PCAN tells us if a gene appears strongly linked to a our patient's phenotype based on known information in public databases. Full details are available in the PCAN manuscript (https://doi.org/10.1186/s12859-016-1401-2). 

We run PCAN for all genes involved in putative fusions.  

#PCAN Code

In the interest of time - <u>do NOT run the next block of code</u>.  A precomputed file (you downloaded it earlier) will be used intead.  Nonetheless, this is fully working code that can be used outside of the demo for your own analysis.  The code is dense, but the inputs (HPO terms, HGNC gene IDs) and outputs (HGNC gene IDs and semantic similarity score ranking per gene) are straightforward.

Feel free to have a look at the code but skip to the next section without running it when you are ready...

In [ ]:
# Just in case...
stop ("You weren't supposed to run this one! Go to the next block of code.")

# Load PCAN library and org.Hs.eg.db (to enable mapping of HGNC gene IDs to Entrez gene IDs used by PCAN)
library(PCAN)
library("org.Hs.eg.db")
args<-commandArgs(TRUE)
# Read in HPO terms for patient's phenotype
hpOfInterest<-scan(file="exostoses_hpo_terms.txt", what="character")

# Read in list of gene IDs (HGNC format) to be scored by PCAN
genIds<-c(fus_unfilt_in$GeneA, fus_unfilt_in$GeneB)
length(genIds)
genIds<-unique(genIds)
length(genIds)

# Main function to run PCAN on a list of phenotype terms and genes

runPCAN <- function(hpOfInterest,genIds){
  if ((length(hpOfInterest)==0)||(length(genIds)==0)){
    #dfout <- data.frame(matrix(ncol =4, nrow = length(genIds)))
    #colnames(dfout) <- c("HGNC_ID", "EntrezGeneID", "candSSScore", "SSScoreRank")
    dfout <- data.frame(matrix(ncol=2, nrow = length(genIds)))
    colnames(dfout) <- c("HGNC_ID", "SSScoreRank")
    return(dfout)
  }
  # Enable conversion of HGNC IDs to Entrez 
  ## Bimap interface
  # Convert the object to a list
  xx <- as.list(org.Hs.egALIAS2EG)
  # Remove pathway identifiers that do not map to any entrez gene id
  xx <- xx[!is.na(xx)]

  ###
  #Read required data sources from text file
  ###
  geneByTrait<-read.table("./PCAN_FINAL_DATA_MAY2019/geneByTrait.txt",sep="\t",header=T)
  traitDef<-read.table("./PCAN_FINAL_DATA_MAY2019/traitDef.txt",sep="\t",header=T)
  geneDef<-as.data.frame(read.table("./PCAN_FINAL_DATA_MAY2019/geneDef.txt",sep="\t",header=T, quote=""))
  hpByTrait<-as.data.frame(read.table("./PCAN_FINAL_DATA_MAY2019/hpByTrait.txt", sep="\t",header=T,quote=""))
  hpDef<-as.data.frame(read.table("./PCAN_FINAL_DATA_MAY2019/hpDef.txt", sep="\t",header=T,quote=""))
  get_descendant<-scan("./PCAN_FINAL_DATA_MAY2019/hp_descendents.txt", sep="\n",what="")
  hp_descendants<-strsplit(get_descendant,";")
  names(hp_descendants)<-sapply(hp_descendants,'[[',1)
  hp_descendants<-lapply(hp_descendants,'[',-1)
  get_ancestor<-scan("./PCAN_FINAL_DATA_MAY2019/hp_ancestors.txt", sep="\n",what="")
  hp_ancestors<-strsplit(get_ancestor,";")
  names(hp_ancestors)<-sapply(hp_ancestors,'[[',1)
  hp_ancestors<-lapply(hp_ancestors,'[',-1)
  geneByHp<-as.data.frame(read.table("./PCAN_FINAL_DATA_MAY2019/geneByHp.txt", sep="\t",header=T,quote=""))
  ###
  ###Defining data sources ENDS
  ###

  info <- unstack(geneByHp, entrez~hp)
  ic <- computeHpIC(info, hp_descendants)
        #Calculate SS scores between HP of interrest and all genes/HP Terms
        hpGeneResnik <- compareHPSets(
        hpSet1=names(ic), hpSet2=hpOfInterest,
        IC=ic,
        ancestors=hp_ancestors,
        method="Resnik",
        BPPARAM=SerialParam()
        )


        ## Group the results by gene
        hpByGene <- unstack(geneByHp, hp~entrez)
        hpMatByGene <- lapply(
        hpByGene,
        function(x){
                hpGeneResnik[x, , drop=FALSE]
        }
        )

        ## Compute the corresponding scores
        resnSss <- unlist(lapply(
        hpMatByGene,
        hpSetCompSummary,
        method="bma", direction="symSim"
        ))
  PCAN_RUN<-function (genId) {
        hgncId<-genId
        if (length(xx[[hgncId]]) ==0) {
          #t_out <-c(hgncId, "NoGene", "NoGene", "NoGene")
          t_out <-c(hgncId, "NoGene")
          return(t_out)
        }
        if (length(xx[[hgncId]])>0) {
          genId_o <- tryCatch(genId_o <- mapIds(org.Hs.eg.db, genId , 'ENTREZID', 'SYMBOL'), error = function(x) x <- paste(mapIds(org.Hs.eg.db, genId , 'ENTREZID', 'ALIAS'),"_alias",sep = ""),silent = T)
          genId <- unlist(strsplit(genId_o,"_"))[1]
          genDis <- traitDef[
          match(
                geneByTrait[which(geneByTrait$entrez==genId), "id"],
                traitDef$id
         ),
          ]
        genHpDef <- hpDef[
        match(
                geneByHp[which(geneByHp$entrez==genId), "hp"],
                hpDef$id
        ),
        ]
        genHp <- genHpDef$id
  if (length(genHp) > 0)  {
        ## Get the score of the gene candidate
        candScore <- resnSss[genId]
        candRank <- sum(resnSss >= candScore)
  }
  if (length(genHp) == 0) {
        candScore<-"NA"
        candRank<-"NA"
  }
  #t_out<-c(hgncId, genId_o, candScore,candRank)
  t_out<-c(hgncId,candRank)
  return(t_out)
  } 
}  ## Outer gene loop
  dfout <- sapply(genIds, function(x) PCAN_RUN(x))
  dfout <- data.frame(t(dfout))
  #colnames(dfout) <- c("HGNC_ID", "EntrezGeneID", "candSSScore", "SSScoreRank")
  colnames(dfout) <- c("HGNC_ID", "SSScoreRank")
  return(dfout)
} ## end function runPCAN

# To generate PCAN rankings
PCANRankings<-runPCAN(hpOfInterest,genIds)


ERROR: ignored

#Load precomputed semantic similarity score rankings

We will load the precomputed PCAN rankings into memory and then use them as if we had just calculated them on-the-fly.  

Then we can have a look at the format of the rankings, and some summary statistics...

In [ ]:
#Print patient phenotpye terms for informational purposes
hpOfInterest<-scan(file="exostoses_hpo_terms.txt", what="character")
cat("Your patient's Human Phenotype Ontology Terms are:\n")
print(hpOfInterest)
# Read precomputed PCAN rankings for every gene from text file (for demo only)
cat("\nLoading precomputed PCAN gene rankings for your patient's phenotype terms\n\n")
PCANRankings<-read.table("PCANRankings.txt", stringsAsFactors=FALSE)
print("Complete - proceeed to next code block")

Your patient's Human Phenotype Ontology Terms are:
 [1] "HP:0002779" "HP:0004322" "HP:0001250" "HP:0011220" "HP:0001302"
 [6] "HP:0000218" "HP:0001263" "HP:0002007" "HP:0100777" "HP:0004325"

Loading precomputed PCAN gene rankings for your patient's phenotype terms

[1] "Complete - proceeed to next code block"



#PCAN outputs

In [ ]:
# Output summarized distribution of PCAN rankings for all genes
summary(as.numeric(as.vector(PCANRankings$SSScoreRank)))
# Preview the table of PCAN rankings for all genes
head(PCANRankings[order(as.numeric(as.vector(PCANRankings$SSScoreRank)),decreasing=FALSE),])

Warning message in summary(as.numeric(as.vector(PCANRankings$SSScoreRank))):
“NAs introduced by coercion”


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
      1     759    1597    1680    2492    3633    7902 

Warning message in order(as.numeric(as.vector(PCANRankings$SSScoreRank)), decreasing = FALSE):
“NAs introduced by coercion”


,HGNC_ID,SSScoreRank
,<chr>,<chr>
TCTN1,TCTN1,1
EXT1,EXT1,2
BRCA2,BRCA2,4
PIK3CA,PIK3CA,7
RB1,RB1,8
FGFR1,FGFR1,9



Despite the dense PCAN code, you can see that the output is very straightforward.  The unexecuted code block contains some commented lines that can be used to generate extra outputs (e.g. actual semantic similarity scores instead of only rankings) but the minimalistic output is sufficient for our purposes.

You will also note that while the rankings skew low there are numerous results in the top ten.  Remember though, that we ran PCAN for every gene in every putative fusion, regardless of other supporting annotations.  Let's add the PCAN rankings to our unfiltered fusion data table and use our annotations to prioritize some results.

In [ ]:
# Ensure PCAN rankings are numeric for selecting high-ranking results later
PCANRankings$SSScoreRank<-as.numeric(PCANRankings$SSScoreRank)

# Add the PCAN ranks for the left and right genes to the unfiltered fusion table and add column names
fus_unfilt_in<-merge(fus_unfilt_in, PCANRankings, by.x="GeneA", by.y="HGNC_ID")
names(fus_unfilt_in)[names(fus_unfilt_in) == 'SSScoreRank'] <- 'LeftSSScoreRank'
fus_unfilt_in<-merge(fus_unfilt_in, PCANRankings, by.x="GeneB", by.y="HGNC_ID")
names(fus_unfilt_in)[names(fus_unfilt_in) == 'SSScoreRank'] <- 'RightSSScoreRank'
# Preview the table
head(fus_unfilt_in)


Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”


,GeneB,GeneA,FusionName,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount,HB_IG_both,HB_IG_either,InGTEx,InInternal,InEither,LeftKnown,RightKnown,LeftSSScoreRank,RightSSScoreRank
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<dbl>
1,5S_rRNA,MED13L,MED13L--5S_rRNA,0,2,chr12:116434917,chrX:68892350,2,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,574,NA
2,A1BG-AS1,IFI16,IFI16--A1BG-AS1,0,2,chr1:158986429,chr19:58865501,2,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,NA
3,AAAS,IRF4,IRF4--AAAS,0,1,chr6:397149,chr12:53702767,1,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,1446
4,AAAS,ARRDC3,ARRDC3--AAAS,0,1,chr5:90670834,chr12:53702095,1,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,1446
5,AACS,CFLAR,CFLAR--AACS,0,10,chr2:202001752,chr12:125621301,10,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,NA
6,AAK1,RPL14P1,RPL14P1--AAK1,0,7,chr12:63359284,chr2:69706164,7,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,NA


#Data exploration

You can see above that we now have a semantic similarity score ranking for the left and right gene in every putative fusion.  There is no right or wrong way to explore the data at this point, and sometimes it is good to output all results to a text file and explore different means of prioritizing results using a spreadsheet viewer.  Based on our own experience though, we will apply some logical means of result prioritization and see what we get.

Let's assume:

i) Any fusion of interest in our patient's phenotype won't involve two globin genes

ii) We expect any pathogenic fusion of interest will not appear in our internal fusion database, or the high confidence GTEx normal tissue database

iii) Any true fusion will have at least 1 supporting read of each read type

iv) Any fusion of phenotypic relevance will have a PCAN ranking higher than 100


In [ ]:
# Count how many fusions do not contain two HB or IG genes 
nrow(subset(fus_unfilt_in, HB_IG_both=="FALSE"))

# Count how many fusions do not a) contain two HB or IG genes and b) occur in either of our databases
nrow(subset(fus_unfilt_in, HB_IG_both=="FALSE" & InEither=="FALSE"))

# Count how many fusions do not a) contain two HB or IG genes, b) occur in either of our databases and 
# c) have at least 1 supporting read of each class
nrow(subset(fus_unfilt_in, HB_IG_both=="FALSE" & InEither=="FALSE" & SpanningFragCount>0 & JunctionReadCount > 0))

# Count how many fusions do not a) contain two HB or IG genes, b) occur in either of our databases, 
# c) have at least 1 supporting read of each class and d) Rank above 100 by PCAN analysis
nrow(subset(fus_unfilt_in, HB_IG_both=="FALSE" & InEither=="FALSE" & SpanningFragCount > 0 & JunctionReadCount > 0 & (LeftSSScoreRank < 100 | RightSSScoreRank < 100)))


[1] 23529

[1] 22111

[1] 558

[1] 2

Utilizing these putative prioritization criteria reduces our number of candidates to 2...

Let's have a closer look...

In [ ]:
# Subset the unfiltered fusion table to those that satisfy the above criteria
subset(fus_unfilt_in, InEither=="FALSE" & SpanningFragCount > 0 & JunctionReadCount > 0 & (LeftSSScoreRank < 100 | RightSSScoreRank < 100) & HB_IG_both=="FALSE")

,GeneB,GeneA,FusionName,JunctionReadCount,SpanningFragCount,LeftBreakpoint,RightBreakpoint,TotalCount,HB_IG_both,HB_IG_either,InGTEx,InInternal,InEither,LeftKnown,RightKnown,LeftSSScoreRank,RightSSScoreRank
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<dbl>
5130,EXT1,SAMD12,SAMD12--EXT1,5,8,chr8:119592954:-,chr8:118849440:-,13,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,NA,2
16955,PIK3R2,SNORA20,SNORA20--PIK3R2,9,1,chr6:160201296:-,chr19:18286148:+,10,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,NA,54


One of the remaining candidates contains a gene ranked at #54 by PCAN.  The other contains one ranked at #2.  Encouraging!

The fusion containing the #2 ranked gene is the only one whose junctions correspond to known exon boundaries too.

Let's look closer at this one...

In [ ]:
# Select the fusion with known exon boundaries an transpose the table for readabilit(ty
t((subset(fus_unfilt_in, InEither=="FALSE" & SpanningFragCount >0 & JunctionReadCount > 0 & (LeftSSScoreRank < 100 | RightSSScoreRank < 100) & (LeftKnown=="TRUE" | RightKnown=="TRUE"))))

,5130
GeneB,EXT1
GeneA,SAMD12
FusionName,SAMD12--EXT1
JunctionReadCount,5
SpanningFragCount,8
LeftBreakpoint,chr8:119592954:-
RightBreakpoint,chr8:118849440:-
TotalCount,13
HB_IG_both,FALSE
HB_IG_either,FALSE


# A SAMD12-EXT1 fusion candidate in a patient with multiple exostoses
We see a fusion candidate that  has a total of 13 supporting reads, doesn't appear in any other samples we have tested, involves two known exon boundaries and one of whose genes (EXT1) is ranked #2 versus all other genes in the ClinVar database.  Mutations in EXT1 are known to be responsible for most cases of multiple exostoses.

At this point some follow-up validation would be necessary, but (spoiler alert) you just discovered the pathogenic fusion responsible for this patient's multiple exostoses within 30 minutes.  Congratulations!


In the image below you can see a schematic diagram of how the fusion is created by to a mosaic deletion of approximately 604KB.  aCGH performed using two separate Agilent arrays reveal the deletion which was initially undetected by clinical testing due to its mosaic nature.



---


![alt text](https://onlinelibrary.wiley.com/cms/asset/c24c613f-fa73-4645-889f-bea9c0c35973/mgg3560-fig-0003-m.png)



---

For more details see the following paper: https://doi.org/10.1002/mgg3.560

Remember that prioritization criteria may need some refinement based on the specifics of the case at-hand, but those shown here are a great starting point.  It is also worth noting that standard filters associated with popular fusion detection algorithms did in fact remove the SAMD12-EXT1 fusion.  Hence our preference for a custom filtering approach.

You have now completed the Fusion Transcript Detection in Rare Disease Tutorial.  Nice work!



---
##FIN !
---